[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/34_speculative_decoding.ipynb)

# 🔴 Hard: Speculative Decoding

Реализуйте **acceptance/rejection step** speculative decoding — метода ускорения генерации, в котором маленькая draft model дешёво предлагает несколько токенов, а большая target model проверяет их параллельно. Корректная схема сохраняет распределение target model, поэтому ускорение не должно менять качество выборки.

## Зачем нужен accept/reject

Draft model задаёт распределение $q(x)$, target model — желаемое $p(x)$. Предложенный токен $x$ принимается с вероятностью

$$a(x)=\min\left(1,\frac{p(x)}{q(x)}\right).$$

Если target считает токен не менее вероятным, чем draft (`p>=q`), он принимается всегда. Если draft переоценила токен, он принимается только с отношением `p/q`. Решения выполняются слева направо: после первого rejection оставшиеся draft tokens уже условны на неверном префиксе и использовать их нельзя.

При rejection нельзя просто семплировать заново из `p`: это исказило бы итоговое распределение, потому что часть массы уже прошла acceptance step. Корректирующее распределение равно

$$p_{\text{res}}(x)=\frac{\max(0,p(x)-q(x))}{\sum_v\max(0,p(v)-q(v))}.$$

Из него выбирается один replacement token, он добавляется в результат, после чего цикл прекращается. Например, если draft token имеет `p=0.2`, `q=0.5`, вероятность принятия равна `0.4`; при rejection новый токен выбирается только из положительной разности target-minus-draft.

> Граница упражнения: функция получает уже готовые вероятности `(K,V)` и реализует только проверку K предложений. Полный алгоритм сам запускает обе модели, поддерживает KV cache и обычно после принятия всего блока добавляет ещё один bonus token target model. По контракту этой задачи результат содержит от 1 до K токенов.

### Signature
```python
def speculative_decode(target_probs, draft_probs, draft_tokens) -> list[int]:
    # target_probs: (K, V) from target (large) model
    # draft_probs: (K, V) from draft (small) model
    # draft_tokens: (K,) tokens sampled by draft model
    # Returns: list of accepted tokens (1 to K)
```

### Algorithm
For each position i = 0, ..., K-1:
1. `ratio = target_probs[i, token_i] / draft_probs[i, token_i]`
2. Accept with probability `min(1, ratio)`
3. If rejected: sample from `normalize(max(0, target - draft))`, append, and stop

## План реализации

1. Идите по draft positions по порядку и извлекайте Python/token index без изменения распределений.
2. Для предложенного token вычислите acceptance probability и сравните её с одним равномерным случайным числом.
3. При acceptance добавьте исходный token и продолжайте.
4. При rejection обрежьте отрицательную разность `p-q`, нормализуйте и сделайте один categorical sample; добавьте его и немедленно остановитесь.

Используйте генераторы PyTorch (`torch.rand`, `torch.multinomial`), чтобы `torch.manual_seed` делал поведение воспроизводимым. Избегайте ненужного преобразования целых распределений в Python/NumPy.

## Быстрые самопроверки

- При `target_probs == draft_probs` все K токенов принимаются.
- Длина результата всегда находится в `[1,K]`.
- После rejection в результат добавляется ровно один replacement и цикл завершается.
- Все возвращённые значения — Python `int` в диапазоне vocabulary.

## Частые ошибки

- Перевернуть отношение и вычислить `q/p`.
- Продолжить проверять draft tokens после первого rejection.
- Семплировать replacement прямо из target distribution вместо положительной разности.
- Забыть нормализацию residual distribution перед `multinomial`.

## Материалы для глубокого изучения

- [Fast Inference from Transformers via Speculative Decoding](https://arxiv.org/abs/2211.17192) — вывод корректной acceptance/rejection схемы и оценка ускорения.
- [Accelerating Large Language Model Decoding with Speculative Sampling](https://arxiv.org/abs/2302.01318) — независимая формулировка speculative sampling.
- [Hugging Face Transformers: assisted generation](https://huggingface.co/docs/transformers/generation_strategies#speculative-decoding) — практическое использование draft/assistant models.

## Вопросы для самопроверки

1. Почему draft token с `p(x)>=q(x)` принимается всегда?
2. Зачем при rejection семплировать из `max(0,p-q)`, а не непосредственно из p?
3. Почему после первого rejection нельзя проверять остаток draft-блока?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def speculative_decode(target_probs, draft_probs, draft_tokens):
    pass  # accept/reject loop

In [ ]:
# 🧪 Debug
torch.manual_seed(0)
probs = torch.softmax(torch.randn(4, 10), dim=-1)
tokens = torch.tensor([2, 5, 1, 8])
print('Perfect draft:', speculative_decode(probs, probs, tokens))
target = torch.softmax(torch.randn(4, 10), dim=-1)
draft = torch.softmax(torch.randn(4, 10), dim=-1)
print('Random draft:', speculative_decode(target, draft, tokens))

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('speculative_decoding')